# 🛡️ CICIDS-2017 — Dataset équilibré 70K (comme le paper)

Paper: *"total number of samples used was 70,343, which balanced the number of samples in each attack category for both datasets"*

70,343 / 15 classes = **4,689 par classe**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q tensorflow numpy pandas scikit-learn matplotlib

In [ ]:
import os, time
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import (Input, Dense, Reshape, Flatten, Concatenate)
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K
from tensorflow.keras import layers, regularizers
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
SEED=42; np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"TF: {tf.__version__}, GPU: {tf.config.list_physical_devices('GPU')}")

In [ ]:
CICIDS_DIR = '/content/drive/MyDrive/cicids2017/MachineLearningCSV/MachineLearningCVE'
csv_files = sorted([f for f in os.listdir(CICIDS_DIR) if f.endswith('.csv')])
print(f"{len(csv_files)} CSV")

dfs = []
for f in csv_files:
    print(f"  {f}...", end=" ")
    try: df = pd.read_csv(os.path.join(CICIDS_DIR,f), encoding='utf-8', low_memory=False)
    except: df = pd.read_csv(os.path.join(CICIDS_DIR,f), encoding='latin-1', low_memory=False)
    df.columns = df.columns.str.strip()
    print(f"{df.shape[0]}")
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
df_all.columns = df_all.columns.str.strip()
label_col = 'Label' if 'Label' in df_all.columns else [c for c in df_all.columns if 'label' in c.lower()][0]
df_all[label_col] = df_all[label_col].str.strip()
df_all.replace([np.inf, -np.inf], np.nan, inplace=True)
df_all.dropna(inplace=True)

# Table 5 features
FEATS = ['Destination Port','Flow Duration','Bwd Packet Length Min',
    'Flow IAT Mean','Flow IAT Max','Fwd IAT Mean',
    'Fwd Packets/s','Bwd Packets/s','Min Packet Length','min_seg_size_forward']
matched = []
for feat in FEATS:
    if feat in df_all.columns: matched.append(feat)
    else:
        found = [c for c in df_all.columns if feat.replace(' ','').lower()==c.replace(' ','').lower()]
        if found: matched.append(found[0])
print(f"\nFeatures: {len(matched)}/10")

y_all = df_all[label_col].copy()
X_all = df_all[matched].apply(pd.to_numeric, errors='coerce')
X_all.dropna(inplace=True); y_all = y_all.loc[X_all.index]

# Train/test split
X_tr_raw, X_te_raw, y_tr, y_te = train_test_split(
    X_all.values, y_all.values, test_size=0.2, random_state=42, stratify=y_all)
scaler = MinMaxScaler()
X_train_all = scaler.fit_transform(X_tr_raw)
X_test_final = scaler.transform(X_te_raw)
y_train_cat = pd.Series(y_tr)
y_test_cat = pd.Series(y_te)
INPUT_DIM = X_train_all.shape[1]

all_labels = sorted(y_train_cat.unique())
N_CLASSES = len(all_labels)
TARGET_PER_CLASS = 70343 // N_CLASSES  # ≈ 4689

print(f"\n✅ Train: {X_train_all.shape}, Test: {X_test_final.shape}")
print(f"Classes: {N_CLASSES}, Target/classe: {TARGET_PER_CLASS}")
print(f"\nDistribution:")
for c in all_labels:
    n = (y_train_cat==c).sum()
    act = 'DOWN' if n>TARGET_PER_CLASS else 'AUG' if n>=10 else 'SKIP'
    print(f"  {c:<25} {n:>8} → {TARGET_PER_CLASS} ({act})")

In [ ]:
# AE-WGAN CICIDS (linear, 2×5)
class AEWGAN_CICIDS:
    def __init__(self):
        self.img_shape=(2,5,1);self.latent_dim=32
        self.flat_dim=int(np.prod(self.img_shape))
        opt=tf.keras.optimizers.Adamax(0.00009)
        self.discriminator=self._bd()
        self.discriminator.compile(loss=self._wl,optimizer=opt,metrics=['accuracy'])
        self.generator=self._bg()
        self.encoder=self._be()
        self.discriminator.trainable=False
        z=Input(shape=(32,));img_=self.generator(z)
        img=Input(shape=(2,5,1));z_=self.encoder(img)
        self.combined=Model([z,img],[self.discriminator([z,img_]),self.discriminator([z_,img])])
        self.combined.compile(loss=[self._wl,self._wl],optimizer=opt)
    def _wl(self,yt,yp): return K.mean(yt*yp)
    def _be(self):
        m=Sequential([Flatten(input_shape=(2,5,1)),Dense(20,activation='linear'),Dense(32)])
        i=Input(shape=(2,5,1));return Model(i,m(i),name='enc')
    def _bg(self):
        m=Sequential([Dense(20,input_dim=32,activation='linear'),Dense(40,activation='linear'),
            Dense(80,activation='linear'),Dense(120,activation='linear'),Dense(160,activation='linear'),
            Dense(self.flat_dim,activation='linear'),Reshape((2,5,1))])
        z=Input(shape=(32,));return Model(z,m(z),name='gen')
    def _bd(self):
        z=Input(shape=(32,));img=Input(shape=(2,5,1))
        d=Concatenate()([z,Flatten()(img)])
        d=Dense(80,activation='linear')(d);d=Dense(40,activation='linear')(d)
        return Model([z,img],Dense(1,activation='sigmoid')(d),name='disc')
    def train(self,X,epochs=101,batch_size=32,verbose=0):
        r=X.reshape(-1,2,5,1);n=r.shape[0]
        if n<batch_size:batch_size=max(n//2,2)
        v=np.ones((batch_size,1));f=np.zeros((batch_size,1))
        for e in range(epochs):
            z=np.random.normal(0,1,(batch_size,32))
            gi=self.generator.predict(z,verbose=0)
            idx=np.random.randint(0,n,batch_size);imgs=r[idx]
            ze=self.encoder.predict(imgs,verbose=0)
            self.discriminator.train_on_batch([ze,imgs],v)
            self.discriminator.train_on_batch([z,gi],f)
            self.combined.train_on_batch([z,imgs],[v,f])
    def generate_batch(self,n):
        return self.generator.predict(np.random.normal(0,1,(n,32)),verbose=0,batch_size=256).reshape(-1,self.flat_dim)

print("✅ AE-WGAN")

In [ ]:
# Construire le dataset équilibré 70K
print("="*60)
print(f"DATASET ÉQUILIBRÉ: {TARGET_PER_CLASS} × {N_CLASSES} = {TARGET_PER_CLASS*N_CLASSES}")
print("="*60)

bal_X=[]; bal_y=[]
start=time.time()

for label in all_labels:
    mask = y_train_cat == label
    X_c = X_train_all[mask.values]; n = X_c.shape[0]

    if n >= TARGET_PER_CLASS:
        idx = np.random.choice(n, TARGET_PER_CLASS, replace=False)
        bal_X.append(X_c[idx]); bal_y.extend([label]*TARGET_PER_CLASS)
        print(f"  [{label}] ⬇️ {n} → {TARGET_PER_CLASS}")
    elif n >= 10:
        ng = TARGET_PER_CLASS - n
        epochs = 300 if n < 500 else 101
        print(f"  [{label}] ⬆️ {n} + {ng} ({epochs} epochs)...", end=" ")
        m = AEWGAN_CICIDS()
        m.train(X_c, epochs=epochs, batch_size=min(32,max(n//2,2)))
        syn = m.generate_batch(ng)
        bal_X.append(np.vstack([X_c, syn])); bal_y.extend([label]*TARGET_PER_CLASS)
        print("✅")
    else:
        print(f"  [{label}] ⏭️ {n}<10")

print(f"--- {(time.time()-start)/60:.1f} min ---")
X_balanced = np.vstack(bal_X)
y_balanced = pd.Series(bal_y)
print(f"\nDataset: {X_balanced.shape[0]} samples")
print(y_balanced.value_counts())

In [ ]:
# Classifieurs CICIDS (tanh)
def build_dt(nc=2): return DecisionTreeClassifier(max_depth=3 if nc==2 else 10,random_state=42)
def build_rf(nc=2): return RandomForestClassifier(100,max_depth=9 if nc==2 else 7,random_state=42,n_jobs=-1)
def build_cnn(dim,nc=2):
    out=1 if nc==2 else nc; act='sigmoid' if nc==2 else 'softmax'
    m=Sequential(name='CNN'); m.add(layers.Input(shape=(dim,1)))
    m.add(layers.Conv1D(40,1,activation='tanh',padding='same'))
    if nc>2: m.add(layers.Conv1D(40,1,activation='tanh',padding='same'))
    m.add(layers.Flatten());m.add(layers.Dense(40,activation='tanh'))
    m.add(layers.Dense(out,activation=act))
    m.compile(optimizer=Adam(1e-3),loss='binary_crossentropy' if nc==2 else 'sparse_categorical_crossentropy',metrics=['accuracy'])
    return m
def build_gru(dim,nc=2):
    out=1 if nc==2 else nc; act='sigmoid' if nc==2 else 'softmax'
    m=Sequential(name='GRU'); m.add(layers.Input(shape=(dim,1)))
    m.add(layers.GRU(40,activation='tanh',return_sequences=True))
    m.add(layers.GRU(40,activation='tanh',return_sequences=True))
    m.add(layers.GRU(40,activation='tanh'))
    if nc==2: m.add(layers.Dense(40,activation='tanh',kernel_regularizer=regularizers.l1(0.005)))
    else: m.add(layers.Dense(40,activation='tanh'))
    m.add(layers.Dense(out,activation=act))
    m.compile(optimizer=Adam(1e-3),loss='binary_crossentropy' if nc==2 else 'sparse_categorical_crossentropy',metrics=['accuracy'])
    return m

def train_eval(name,clf,X_tr,y_tr,X_te,y_te):
    if name in ['CNN','GRU']:
        X3=X_tr.reshape(-1,X_tr.shape[1],1);Xt3=X_te.reshape(-1,X_te.shape[1],1)
        es=EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)
        clf.fit(X3,y_tr,epochs=50,batch_size=256,validation_split=0.1,callbacks=[es],verbose=0)
        yp=clf.predict(Xt3,verbose=0)
        y_pred=(yp.flatten()>0.5).astype(int) if yp.shape[-1]==1 else np.argmax(yp,axis=1)
    else: clf.fit(X_tr,y_tr);y_pred=clf.predict(X_te)
    avg='binary' if len(np.unique(y_te))==2 else 'weighted'
    return {'classifier':name,'accuracy':accuracy_score(y_te,y_pred),
            'precision':precision_score(y_te,y_pred,average=avg,zero_division=0),
            'recall':recall_score(y_te,y_pred,average=avg,zero_division=0),
            'f1_score':f1_score(y_te,y_pred,average=avg,zero_division=0)}

print("✅")

In [ ]:
# Classification
def run_all(X_tr,y_cat,X_te,y_te_cat,dim,binary,label):
    le=LabelEncoder()
    if binary: y_tr=(y_cat!='BENIGN').astype(int).values;y_te=(y_te_cat!='BENIGN').astype(int).values
    else:
        le.fit(list(set(y_cat.unique())|set(y_te_cat.unique())))
        y_tr=le.transform(y_cat);y_te=le.transform(y_te_cat)
    nc=2 if binary else len(np.unique(y_tr))
    tag='BIN' if binary else 'MULTI'
    print(f"\n  [{label} {tag}] Classes:{nc} Train:{X_tr.shape[0]}")
    results=[]
    for name,builder in [('DT',lambda:build_dt(nc)),('RF',lambda:build_rf(nc)),
                         ('CNN',lambda:build_cnn(dim,nc)),('GRU',lambda:build_gru(dim,nc))]:
        print(f"    {name}...",end=" ")
        r=train_eval(name,builder(),X_tr,y_tr,X_te,y_te)
        print(f"Acc={r['accuracy']:.4f} F1={r['f1_score']:.4f}")
        results.append(r)
    return results

# Baseline (full dataset)
base_bin=run_all(X_train_all,y_train_cat,X_test_final,y_test_cat,INPUT_DIM,True,'Baseline')
base_multi=run_all(X_train_all,y_train_cat,X_test_final,y_test_cat,INPUT_DIM,False,'Baseline')

# Balanced 70K
bal_bin=run_all(X_balanced,y_balanced,X_test_final,y_test_cat,INPUT_DIM,True,'Bal70K')
bal_multi=run_all(X_balanced,y_balanced,X_test_final,y_test_cat,INPUT_DIM,False,'Bal70K')

In [ ]:
# Comparaison avec le paper
print("\n" + "="*70)
print("  COMPARAISON: Baseline vs Balanced 70K vs Paper")
print("="*70)

paper_bin = {'DT':0.84,'RF':0.88,'CNN':0.98,'GRU':0.98}
paper_multi = {'DT':0.52,'RF':0.63,'CNN':0.65,'GRU':0.69}

print(f"\n  --- BINAIRE ---")
print(f"  {'Clf':<6} {'Base F1':>9} {'Bal70K F1':>10} {'Paper F1':>10} {'Écart Paper':>12}")
print(f"  {'─'*50}")
for b,bl in zip(base_bin,bal_bin):
    n=b['classifier']
    p=paper_bin.get(n,0)
    gap=bl['f1_score']-p
    print(f"  {n:<6} {b['f1_score']:>9.4f} {bl['f1_score']:>10.4f} {p:>10.2f} {'+'if gap>=0 else ''}{gap:>11.4f}")

print(f"\n  --- MULTICLASSE ---")
print(f"  {'Clf':<6} {'Base F1':>9} {'Bal70K F1':>10} {'Paper F1':>10} {'Écart Paper':>12}")
print(f"  {'─'*50}")
for b,bl in zip(base_multi,bal_multi):
    n=b['classifier']
    p=paper_multi.get(n,0)
    gap=bl['f1_score']-p
    print(f"  {n:<6} {b['f1_score']:>9.4f} {bl['f1_score']:>10.4f} {p:>10.2f} {'+'if gap>=0 else ''}{gap:>11.4f}")

# Chart
fig,axes=plt.subplots(1,2,figsize=(14,6))
fig.suptitle('CICIDS-2017: Baseline vs Balanced 70K vs Paper',fontsize=14,fontweight='bold')
for idx,(task,br,blr,pv) in enumerate([('Binary',base_bin,bal_bin,paper_bin),('Multiclass',base_multi,bal_multi,paper_multi)]):
    ax=axes[idx]
    clfs=[r['classifier'] for r in br]; x=np.arange(len(clfs)); w=0.25
    ax.bar(x-w,[r['f1_score'] for r in br],w,label='Baseline (full)',color='#3498db',alpha=0.8)
    ax.bar(x,[r['f1_score'] for r in blr],w,label='Balanced 70K',color='#e74c3c',alpha=0.8)
    ax.bar(x+w,[pv.get(c,0) for c in clfs],w,label='Paper',color='#f39c12',alpha=0.8)
    ax.set_xlabel('Classifier');ax.set_ylabel('F1');ax.set_title(task)
    ax.set_xticks(x);ax.set_xticklabels(clfs);ax.legend();ax.set_ylim(0,1.1);ax.grid(axis='y',alpha=0.3)
plt.tight_layout()
plt.savefig('results_cicids_70K.png',dpi=150,bbox_inches='tight')
plt.show()

# Save
rows=[]
for r in base_bin: rows.append({**r,'type':'Baseline','task':'Binary'})
for r in bal_bin: rows.append({**r,'type':'Balanced70K','task':'Binary'})
for r in base_multi: rows.append({**r,'type':'Baseline','task':'Multiclass'})
for r in bal_multi: rows.append({**r,'type':'Balanced70K','task':'Multiclass'})
df=pd.DataFrame(rows)
df.to_csv('results_cicids_70K.csv',index=False)
import shutil
shutil.copy('results_cicids_70K.csv','/content/drive/MyDrive/')
shutil.copy('results_cicids_70K.png','/content/drive/MyDrive/')
print("\n✅ Sauvegardé")
print(df.to_string(index=False))